In [203]:
import pandas as pd
import sqlite3
import requests
import io
from urllib.parse import urlencode

In [204]:
base_url = 'https://cloud-api.yandex.net/v1/disk/public/resources/download?'
public_key = 'https://disk.yandex.by/d/_7Vxv01EcFTiyg'

final_url = base_url + urlencode(dict(public_key=public_key))
response = requests.get(final_url)
download_url = response.json()['href']

download_response = requests.get(download_url)
df = pd.read_csv(io.BytesIO(download_response.content))
df

,user_id,timestamp,amount,ip,country,device_id,status,is_fraud
0,user_16,2023-01-03 17:33:00,5.96,192.168.154.153,US,device_174,success,0
1,user_44,2023-01-03 19:57:00,104.39,192.168.124.223,RU,device_217,success,0
2,user_33,2023-01-02 22:15:00,3.04,192.168.195.50,UK,device_1,success,0
3,user_21,2023-01-02 11:15:00,38.58,192.168.219.216,UK,device_281,success,0
4,user_8,2023-01-03 11:27:00,46.17,192.168.221.86,US,device_236,success,0
...,...,...,...,...,...,...,...,...
5135,user_1,2023-01-01 05:10:00,1.02,192.168.16.169,RU,device_224,success,0
5136,user_36,2023-01-02 04:29:00,159.23,192.168.156.163,UK,device_204,success,0
5137,user_99,2023-01-02 18:16:00,81.62,192.168.212.119,FR,device_257,success,0
5138,user_76,2023-01-01 19:47:00,34.23,192.168.215.182,RU,device_59,success,0


In [205]:
conn = sqlite3.connect(":memory:")

In [206]:
df.to_sql("transactions", conn, index=False, if_exists="replace")

5140

# **Выгрузить топ 10 пользователей по количеству операций за час**

In [207]:
Top_users_per_hour = """
SELECT user_id,
strftime('%Y-%m-%d %H', timestamp) as hour,
COUNT(*) as cnt
FROM transactions
GROUP BY user_id, hour
ORDER BY cnt DESC
LIMIT 10;
"""
# Для PostgreSQL - date_trunc('hour', timestamp) as hour
high_freq = pd.read_sql(Top_users_per_hour, conn)
high_freq

,user_id,hour,cnt
0,user_64,2023-01-01 11,22
1,user_67,2023-01-01 16,21
2,user_42,2023-01-03 02,20
3,user_44,2023-01-01 15,20
4,user_59,2023-01-03 04,20
5,user_22,2023-01-03 12,7
6,user_43,2023-01-02 15,7
7,user_12,2023-01-01 13,6
8,user_23,2023-01-02 20,6
9,user_17,2023-01-01 14,5


# **Найти пользователей совершавших транзакции из 3+ разных стран за сутки**

In [208]:
Top_users_per_country = """
SELECT
    user_id,
    DATE(timestamp) AS date,
    COUNT(DISTINCT country) AS country_cnt,
    COUNT(*) as txn_cnt
FROM transactions
GROUP BY user_id, DATE(timestamp)
HAVING COUNT(DISTINCT country) >= 3
ORDER BY country_cnt DESC, txn_cnt DESC;
"""
multi_country = pd.read_sql(Top_users_per_country, conn)
multi_country

,user_id,date,country_cnt,txn_cnt
0,user_64,2023-01-01,5,36
1,user_59,2023-01-03,5,33
2,user_74,2023-01-01,5,31
3,user_83,2023-01-02,5,29
4,user_41,2023-01-03,5,29
...,...,...,...,...
295,user_20,2023-01-01,4,10
296,user_24,2023-01-02,4,8
297,user_73,2023-01-03,3,16
298,user_95,2023-01-01,3,13


# **Выявить операции с нехарактерно высокой суммой (например, >3 стандартных отклонений от среднего для клиента)**

In [209]:
# тк у меня нет PostgreSQL, в котором STDDEV прямо из коробки, работаю через pandas-костыль

df['avg'] = df.groupby('user_id')['amount'].transform('mean')
df['std'] = df.groupby('user_id')['amount'].transform('std')

df['is_outlier'] = df['amount'] > (df['avg'] + 3 * df['std'])

df


# для PostgreSQL
# SELECT *
# FROM (
#     SELECT
#         user_id,
#         timestamp,
#         amount,
#         AVG(amount) OVER (PARTITION BY user_id) as avg_amount,
#         STDDEV(amount) OVER (PARTITION BY user_id) as std_amount
#     FROM transactions
# ) t
# WHERE amount > avg_amount + 3 * std_amount
# AND COUNT(*) OVER (PARTITION BY user_id) > 5;

,user_id,timestamp,amount,ip,country,device_id,status,is_fraud,avg,std,is_outlier
0,user_16,2023-01-03 17:33:00,5.96,192.168.154.153,US,device_174,success,0,47.837736,49.401542,False
1,user_44,2023-01-03 19:57:00,104.39,192.168.124.223,RU,device_217,success,0,38.034325,49.470387,False
2,user_33,2023-01-02 22:15:00,3.04,192.168.195.50,UK,device_1,success,0,61.404048,64.773846,False
3,user_21,2023-01-02 11:15:00,38.58,192.168.219.216,UK,device_281,success,0,45.826727,38.182360,False
4,user_8,2023-01-03 11:27:00,46.17,192.168.221.86,US,device_236,success,0,42.990000,63.613651,False
...,...,...,...,...,...,...,...,...,...,...,...
5135,user_1,2023-01-01 05:10:00,1.02,192.168.16.169,RU,device_224,success,0,51.641053,50.318854,False
5136,user_36,2023-01-02 04:29:00,159.23,192.168.156.163,UK,device_204,success,0,50.579464,49.536855,False
5137,user_99,2023-01-02 18:16:00,81.62,192.168.212.119,FR,device_257,success,0,54.445667,48.573577,False
5138,user_76,2023-01-01 19:47:00,34.23,192.168.215.182,RU,device_59,success,0,48.998772,47.859319,False


In [210]:
df_outliers = df[df['is_outlier'] == True]
df_outliers

,user_id,timestamp,amount,ip,country,device_id,status,is_fraud,avg,std,is_outlier
99,user_83,2023-01-01 01:00:00,214.88,192.168.240.54,FR,device_37,success,0,51.318209,52.470904,True
145,user_99,2023-01-02 06:39:00,202.46,192.168.69.77,UK,device_123,success,0,54.445667,48.573577,True
182,user_90,2023-01-03 12:50:00,166.24,192.168.3.153,FR,device_203,success,0,43.648906,40.135700,True
230,user_42,2023-01-03 07:07:00,159.72,192.168.120.237,RU,device_270,success,0,30.619463,41.085852,True
277,user_25,2023-01-02 22:39:00,315.30,192.168.88.211,UK,device_159,success,0,50.495417,53.905178,True
...,...,...,...,...,...,...,...,...,...,...,...
4984,user_49,2023-01-03 15:50:00,254.03,192.168.37.141,US,device_34,success,0,48.794043,53.447627,True
5009,user_54,2023-01-02 11:21:00,294.54,192.168.90.252,FR,device_291,success,0,50.163043,60.854246,True
5029,user_19,2023-01-03 09:22:00,240.02,192.168.144.142,UK,device_189,success,0,54.761475,47.558325,True
5093,user_47,2023-01-02 21:38:00,211.15,192.168.205.39,DE,device_239,success,0,44.376905,42.665839,True


# **Проанализировать кейсы: быстрое перебирание CVV-кодов (Брутфорс)**

In [211]:
bruteforce = """
WITH base AS (
    SELECT
        user_id,
        timestamp,
        status,
        LAG(timestamp) OVER (PARTITION BY user_id ORDER BY timestamp) as prev_ts
    FROM transactions
),

flags AS (
    SELECT *,
        (julianday(timestamp) - julianday(prev_ts)) * 86400 as diff_sec
    FROM base
)

SELECT
    user_id,
    COUNT(*) as attempts,
    SUM(CASE WHEN status = 'decline' THEN 1 ELSE 0 END) as declines
FROM flags
WHERE diff_sec < 10
GROUP BY user_id
HAVING COUNT(*) > 10
   AND COUNT(CASE WHEN status = 'decline' THEN 1 END) > 8
ORDER BY attempts DESC;
"""
bruteforce_result = pd.read_sql(bruteforce, conn)
bruteforce_result

,user_id,attempts,declines
0,user_64,13,13
1,user_42,13,13
2,user_67,12,12
3,user_59,12,12
4,user_44,12,12


# **Выявить подозрительные смены устройств и IP-адресов**

In [212]:
ip_change = """
SELECT
user_id,
DATE(timestamp) AS date,
COUNT(DISTINCT ip) AS ip_cnt
FROM transactions
GROUP BY user_id, DATE(timestamp)
HAVING COUNT(DISTINCT ip) > 3;
"""
ip_change_result = pd.read_sql(ip_change, conn)
ip_change_result

,user_id,date,ip_cnt
0,user_0,2023-01-01,15
1,user_0,2023-01-02,16
2,user_0,2023-01-03,20
3,user_1,2023-01-01,20
4,user_1,2023-01-02,22
...,...,...,...
295,user_98,2023-01-02,28
296,user_98,2023-01-03,10
297,user_99,2023-01-01,25
298,user_99,2023-01-02,16


In [213]:
device_change = """
SELECT
user_id,
DATE(timestamp) AS date,
COUNT(DISTINCT device_id) AS device_id_cnt
FROM transactions
GROUP BY user_id, DATE(timestamp)
HAVING COUNT(DISTINCT device_id) > 3;
"""
device_change_result = pd.read_sql(device_change, conn)
device_change_result

,user_id,date,device_id_cnt
0,user_0,2023-01-01,14
1,user_0,2023-01-02,16
2,user_0,2023-01-03,18
3,user_1,2023-01-01,19
4,user_1,2023-01-02,22
...,...,...,...
295,user_98,2023-01-02,27
296,user_98,2023-01-03,9
297,user_99,2023-01-01,23
298,user_99,2023-01-02,16


In [214]:
# самое интересное - собираем результаты исследований вместе
# для начала нужно задать вес каждому из исследованных признаков

weights = {
    'high_freq': 3,
    'multi_country': 2,
    'amount_outlier': 2,
    'bruteforce': 5,
    'ip_change': 2,
    'device_change': 3
}

In [215]:
# база пользователей
users = df[['user_id']].drop_duplicates()
users['score'] = 0
users['reasons'] = ''

In [216]:
def add_reason(mask, reason, weight):
    users.loc[mask, 'score'] += weight
    users.loc[mask, 'reasons'] = users.loc[mask, 'reasons'] + reason + ' | '

# берёт только строки, где mask = True (потому что пользователь попал в выборку) и увеличивает score на weight
mask = users['user_id'].isin(high_freq['user_id'])
add_reason(mask, 'high_freq', weights['high_freq'])

mask = users['user_id'].isin(multi_country['user_id'])
add_reason(mask, 'multi_country', weights['multi_country'])

mask = users['user_id'].isin(df_outliers['user_id'])
add_reason(mask, 'amount_outlier', weights['amount_outlier'])

mask = users['user_id'].isin(bruteforce_result['user_id'])
add_reason(mask, 'bruteforce', weights['bruteforce'])

mask = users['user_id'].isin(ip_change_result['user_id'])
add_reason(mask, 'ip_change', weights['ip_change'])

mask = users['user_id'].isin(device_change_result['user_id'])
add_reason(mask, 'device_change', weights['device_change'])

In [217]:
threshold = 5

users['is_fraud'] = (users['score'] >= threshold).astype(int)

In [218]:
users.sort_values('score', ascending=False)

,user_id,score,reasons,is_fraud
1,user_44,17,high_freq | multi_country | amount_outlier | b...,1
19,user_67,17,high_freq | multi_country | amount_outlier | b...,1
36,user_59,17,high_freq | multi_country | amount_outlier | b...,1
76,user_42,17,high_freq | multi_country | amount_outlier | b...,1
278,user_64,17,high_freq | multi_country | amount_outlier | b...,1
...,...,...,...,...
109,user_26,7,multi_country | ip_change | device_change |,1
151,user_6,7,multi_country | ip_change | device_change |,1
147,user_86,7,multi_country | ip_change | device_change |,1
189,user_28,7,multi_country | ip_change | device_change |,1


# **Правила:**

IF:
у пользователя > 10 транзакций за 1 минуту

THEN:
пометить как fraud

Почему:
паттерн характерен для автоматизированного перебора (bruteforce)



---




IF:
у пользователя несколько device_id за короткий период

THEN:
пометить как подозрительно

Почему:
возможно перехват аккаунта



---



IF: amount > avg + 3 * std для пользователя

THEN: пометить как fraud

Почему: выброс относительно нормального поведения



---



IF: ≥ 3 разных стран за сутки

THEN: пометить как подозрительно

Почему: физически невозможно - VPN / компрометация